# 3. Data statistics

*Author: Phạm Tấn Nhật Thịnh · code: `src/04_monitor.ipynb`, `src/05_run_pipeline.ipynb`*

This chapter describes what the crawl produced for the 20,000 assigned IDs: how many papers and versions
were scraped and what failed (3.1), how large a paper is before and after the figures are removed (3.2),
what the reference metadata looks like (3.3), and how the corpus breaks down by version count, subject and
date (3.4).

**Every number, table and figure below is computed when the book is built**, from the files in `data/`
that `tools/export_report_data.py` derives from the crawl's logs and state (the code of each cell can be
unfolded). All figures are interactive Plotly charts: hover for exact values, drag to zoom, double-click to
reset, and click legend entries to filter series. The populations differ from section to section and are
named each time:

| Level | Source | One row per |
|---|---|---|
| paper | `papers.csv` (pipeline log), `metadata.csv` (discovery manifest) | assigned ID |
| version | `versions.csv` (download receipts) | source version that was processed |
| request event | `download_events.csv` (`errors.jsonl`, `throttles.jsonl`) | failed or refused request |
| reference | `reference_fields.json`, `references_log.csv` | saved reference / logged request |

The four assigned ranges are called **R1–R4** in the order of `ID_RANGES` in `src/00_config.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import report_data as rd

rd.check()
bench = rd.benchmark()
papers, meta, versions, events = rd.papers(), rd.metadata(), rd.versions(), rd.events()
RANGE_LABEL = {name: f"{name}  {first}–{last}" for name, (first, last) in rd.RANGES.items()}


def pct(part, whole, digits=2):
    return f"{100 * part / whole:.{digits}f}%" if whole else "–"


def with_total(frame, label="Total"):
    total = {frame.columns[0]: label, **frame.sum(numeric_only=True).to_dict()}
    return pd.concat([frame, pd.DataFrame([total])], ignore_index=True)

## 3.1 Scraping success rate and failure breakdown

**Definition.** A paper is *scraped successfully* when **every one of its versions has a download
receipt**: the version was requested, unpacked and reduced to its `.tex`/`.bib` files, or it was found to
have no LaTeX source (PDF or Word only) and keeps an empty `vN/` folder, as the statement asks. A paper with
even one unprocessed version is counted as **incomplete**, although its other versions are in the corpus.

$$
\text{success rate} = \frac{\#\{\text{papers whose versions all have a receipt}\}}{\#\{\text{assigned IDs}\}}
$$

Reference metadata is evaluated separately in section 3.3 and does not enter this rate.

In [ ]:
by_range = papers.groupby("range").agg(
    assigned=("id", "size"),
    on_arxiv=("versions", lambda s: int((s > 0).sum())),
    scraped=("outcome", lambda s: int(s.isin(["latex", "mixed", "pdf_only"]).sum())),
    with_latex=("outcome", lambda s: int(s.isin(["latex", "mixed"]).sum())),
    pdf_only=("outcome", lambda s: int((s == "pdf_only").sum())),
    incomplete=("outcome", lambda s: int((s == "incomplete").sum())),
).reset_index()
by_range["range"] = by_range["range"].map(RANGE_LABEL)
t31 = with_total(by_range)
t31["success rate"] = 100 * t31["scraped"] / t31["assigned"]
t31.columns = ["Range", "Assigned IDs", "Found on arXiv", "Scraped successfully", "… with LaTeX source",
               "… without LaTeX (empty record)", "Incomplete", "Success rate"]
rd.table(t31, "Table 3.1. Paper-level outcome of the crawl by ID range.", {"Success rate": "{:.2f}%"})

total = t31.iloc[-1]
assert total["Scraped successfully"] == bench["papers_scraped"], "papers.csv and benchmark.json disagree"
rd.md(
    f"All **{int(total['Assigned IDs']):,}** assigned IDs exist on arXiv "
    f"({int(total['Assigned IDs'] - total['Found on arXiv']):,} unused IDs), so the success rate over the assigned "
    f"IDs and over the papers found is the same: **{total['Success rate']:.2f}%** "
    f"({int(total['Scraped successfully']):,} papers). **{int(total['… with LaTeX source']):,}** papers have LaTeX "
    f"for at least one version; **{int(total['… without LaTeX (empty record)']):,}** have none and are kept as empty "
    f"records; **{int(total['Incomplete']):,}** are incomplete. The rate is nearly the same in the four ranges "
    f"({t31['Success rate'][:-1].min():.2f}%–{t31['Success rate'][:-1].max():.2f}%).")

In [ ]:
order = ["latex", "mixed", "pdf_only", "incomplete"]
labels = {"latex": "LaTeX for every version", "mixed": "LaTeX for some versions",
          "pdf_only": "no LaTeX source (empty record)", "incomplete": "incomplete (a version missing)"}
counts = pd.crosstab(papers["range"], papers["outcome"]).reindex(columns=order, fill_value=0)
totals_r = counts.sum(axis=1)

fig = make_subplots(
    rows=1, cols=2, column_widths=[0.58, 0.42], horizontal_spacing=0.10,
    subplot_titles=("All assigned IDs", "Without 'LaTeX for every version'"),
)
for col_idx, cols in ((1, order), (2, order[1:])):
    for name in cols:
        vals = counts[name]
        shares = 100 * vals / totals_r
        fig.add_trace(
            go.Bar(
                y=counts.index.tolist(),
                x=vals.tolist(),
                name=labels[name],
                legendgroup=name,
                showlegend=(col_idx == 1),
                orientation="h",
                marker_color=rd.SERIES[order.index(name)],
                customdata=np.stack([shares], axis=-1),
                hovertemplate="%{y} · " + labels[name] + "<br><b>%{x:,}</b> papers (%{customdata[0]:.2f}%)<extra></extra>",
            ),
            row=1, col=col_idx,
        )
fig.update_layout(
    barmode="stack",
    title="Paper-level outcome by ID range",
    legend=dict(orientation="h", yanchor="top", y=-0.20, xanchor="center", x=0.5),
)
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_xaxes(title_text="Papers", row=1, col=1)
fig.update_xaxes(title_text="Papers", row=1, col=2)
rd.show(fig, height=340, margin=dict(l=45, r=20, t=65, b=80))
rd.figure_caption("Figure 3.1. Paper-level outcome by ID range. Right: the same bars without the dominant "
                  "class, to make the three small classes visible.")

### Version-level outcomes

Paper-level success hides what happened to each version, so the same result is broken down per version.
The number of versions of each paper comes from entry discovery; a version is *processed* when it has a
receipt, and an unprocessed version is classified by the `error_kind` of its **last** failed request.

In [ ]:
expected = int(meta["versions"].sum())
status = versions["status"].value_counts()
done_keys = set(zip(versions["id"], versions["version"]))
failed = events[events["log"] == "errors"].sort_values("at").groupby(["id", "version"]).tail(1)
failed = failed[[key not in done_keys for key in zip(failed["id"], failed["version"])]]
unprocessed = failed["error_kind"].value_counts()
KIND_TEXT = {
    "NOT_FOUND": "HTTP 404/410: could not be downloaded, kept as empty record in M2",
    "CORRUPT_ARCHIVE": "archive cannot be unpacked (not_downloaded in M2)",
    "REJECTED_ARCHIVE": "archive rejected by extractor safety check (not_downloaded in M2)",
    "TRANSFER_CUT": "transfer cut and not recovered",
}
rows = [("processed: `.tex`/`.bib` kept (`ok`)", status.get("ok", 0)),
        ("processed: no LaTeX source, empty record (`empty_source`)", status.get("empty_source", 0))]
rows += [(f"not processed: {KIND_TEXT.get(kind, kind)} (`{kind}`)", n) for kind, n in unprocessed.items()]
logged = sum(n for _, n in rows)
if expected - logged:
    rows.append(("not processed: no request logged", expected - logged))
t32 = pd.DataFrame(rows, columns=["Outcome of the version", "Versions"])
t32["Share"] = 100 * t32["Versions"] / expected
t32 = pd.concat([t32, pd.DataFrame([("**Versions reported by entry discovery**", expected, 100.0)],
                                   columns=t32.columns)], ignore_index=True)
rd.table(t32, "Table 3.2. Outcome of every source version.", {"Share": "{:.2f}%"})

missing = expected - len(versions)
not_found = int(unprocessed.get("NOT_FOUND", 0))
latest = meta.set_index("id")["versions"]
is_latest = sum(latest[i] == v for i, v in zip(failed["id"], failed["version"]))
rd.md(
    f"**{len(versions):,}** of the **{expected:,}** versions were processed in Milestone 1 ({pct(len(versions), expected)}): "
    f"**{status.get('ok', 0):,}** `ok` and **{status.get('empty_source', 0):,}** PDF-only `empty_source`. "
    f"The remaining **{missing:,}** versions have no Milestone 1 receipt and belong to the {int(total['Incomplete']):,} "
    f"incomplete papers: **{not_found:,}** persistent HTTP 404 versions (`NOT_FOUND`) and **{missing - not_found:,}** "
    f"versions whose archives repeatedly failed extraction (`REJECTED_ARCHIVE` / `CORRUPT_ARCHIVE`). "
    f"Of the {len(failed):,} unprocessed versions, {is_latest:,} are the latest version of their paper and "
    f"{len(failed) - is_latest:,} are earlier revisions.")

### Source HTTP 404 versions and the 3 unextractable archives

Following the instructor's guidance (Q&A M1 #23), versions whose `export.arxiv.org/src/<id>v<n>` endpoint
returns **HTTP 404** are recorded as *could not be downloaded (HTTP 404)* and carried forward as **empty
records**—the same way as PDF-only versions—without querying external mirrors. They represent real revisions
listed by the arXiv API whose source package was hidden by the authors or failed compilation.

In [ ]:
nf_events = events[events["error_kind"] == "NOT_FOUND"].copy()
nf_summary = (
    nf_events.sort_values("at")
    .groupby(["id", "version"], as_index=False)
    .agg(attempts=("at", "size"), first_at=("at", "min"), last_at=("at", "max"))
)
nf_summary["range"] = nf_summary["id"].map(rd.range_of)
nf_summary["version_id"] = nf_summary["id"] + "v" + nf_summary["version"].astype(str)

by_rng = nf_summary["range"].value_counts().reindex(list(rd.RANGES), fill_value=0)
by_ver = nf_summary["version"].value_counts().sort_index()

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.14,
    subplot_titles=("HTTP 404 versions by ID range", "HTTP 404 versions by version number"),
)
fig.add_trace(
    go.Bar(
        x=by_rng.index.tolist(), y=by_rng.values.tolist(),
        marker_color=rd.ORANGE, text=by_rng.values.tolist(), textposition="outside",
        hovertemplate="%{x}: <b>%{y}</b> versions<extra></extra>", showlegend=False,
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(
        x=[f"v{v}" for v in by_ver.index], y=by_ver.values.tolist(),
        marker_color=rd.BLUE, text=by_ver.values.tolist(), textposition="outside",
        hovertemplate="Version %{x}: <b>%{y}</b> versions<extra></extra>", showlegend=False,
    ),
    row=1, col=2,
)
fig.update_layout(title="Breakdown of the 100 persistent /src/ HTTP 404 versions")
fig.update_xaxes(showgrid=False, row=1, col=1)
fig.update_xaxes(showgrid=False, row=1, col=2)
fig.update_yaxes(title_text="Versions", range=[0, by_rng.max() * 1.22], row=1, col=1)
fig.update_yaxes(title_text="Versions", range=[0, by_ver.max() * 1.18], row=1, col=2)
rd.show(fig, height=340)
rd.figure_caption("Figure 3.2. Distribution of the persistent HTTP 404 source versions by assigned ID range (left) "
                  "and by revision number (right).")

other_fail = failed[failed["error_kind"] != "NOT_FOUND"].sort_values(["id", "version"])
other_details = []
for r in other_fail.itertuples():
    n_att = int(((events["id"] == r.id) & (events["version"] == r.version) & (events["log"] == "errors")).sum())
    other_details.append(f"`{r.id}v{r.version}` (`{r.error_kind}`, {n_att} attempts)")

rd.md(
    f"In total, **{len(nf_summary):,} distinct versions** across **{nf_summary['id'].nunique():,} papers** "
    f"returned HTTP 404 on every retry (**{len(nf_events):,} `NOT_FOUND` events** across sessions; median "
    f"{nf_summary['attempts'].median():.0f} attempts per version, up to {int(nf_summary['attempts'].max())}, "
    f"last retried on {nf_summary['last_at'].max():%d %b %Y}). Most are revision `v2` "
    f"({int(by_ver.get(2, 0)):,} of {len(nf_summary):,}), spread evenly across the four ranges "
    f"({', '.join(f'{k}: {int(v)}' for k, v in by_rng.items())}). "
    f"The remaining **{len(other_fail)} versions** without an M1 receipt are {', '.join(other_details)}: "
    f"each was downloaded in 3 separate runs and consistently failed extraction or safety validation. "
    f"In [Milestone 2 (Table 3.4)](../milestone2/statistics.ipynb), all **{status.get('empty_source', 0):,}** "
    f"PDF-only versions, **{len(nf_summary):,}** `not_found` (HTTP 404) versions and **{len(other_fail)}** "
    f"`not_downloaded` versions are preserved together as **{status.get('empty_source', 0) + len(nf_summary) + len(other_fail):,}** "
    f"`empty_source` records.")

nf_table = nf_summary[["version_id", "range", "version", "attempts", "first_at", "last_at"]].copy()
nf_table["first_at"] = nf_table["first_at"].dt.strftime("%Y-%m-%d %H:%M")
nf_table["last_at"] = nf_table["last_at"].dt.strftime("%Y-%m-%d %H:%M")
nf_table.columns = ["Version", "Range", "v", "Attempts", "First 404 (UTC)", "Last 404 (UTC)"]
rd.collapsible_table(nf_table, f"Show all {len(nf_table)} persistent /src/ HTTP 404 versions (click to expand)")

### Failed and refused requests

`errors.jsonl` records every request that **failed** (the version is asked again in the next run) and
`throttles.jsonl` every request arXiv **refused** or left unanswered (the version goes back to the end of
the queue). These are *events*, not versions: one version can fail several times before it succeeds.

In [ ]:
ev = events.copy()
ev["recovered"] = [key in done_keys for key in zip(ev["id"], ev["version"])]
by_kind = ev.groupby("error_kind").agg(
    events=("id", "size"),
    versions=("id", lambda s: len(set(zip(s, ev.loc[s.index, "version"])))),
).sort_values("events", ascending=True)
recovered = ev[ev["recovered"]].groupby("error_kind").apply(
    lambda d: len(set(zip(d["id"], d["version"]))), include_groups=False)
by_kind["recovered"] = recovered.reindex(by_kind.index).fillna(0).astype(int)

fig = go.Figure(go.Bar(
    y=by_kind.index.tolist(),
    x=by_kind["events"].tolist(),
    orientation="h",
    marker_color=rd.BLUE,
    text=[f"{e:,} events · {v:,} versions ({r:,} recovered)"
          for e, v, r in zip(by_kind["events"], by_kind["versions"], by_kind["recovered"])],
    textposition="outside",
    customdata=np.stack([by_kind["versions"], by_kind["recovered"]], axis=-1),
    hovertemplate="<b>%{y}</b><br>Events: %{x:,}<br>Distinct versions: %{customdata[0]:,}<br>"
                  "Later processed: %{customdata[1]:,}<extra></extra>",
))
fig.update_layout(title="Failed and refused source requests by error kind")
fig.update_xaxes(title_text="Logged events", range=[0, by_kind["events"].max() * 1.55])
fig.update_yaxes(showgrid=False)
rd.show(fig, height=max(300, 44 * len(by_kind) + 100), margin=dict(l=145, r=25, t=55, b=45))
rd.figure_caption("Figure 3.3. Events in `errors.jsonl` and `throttles.jsonl` by `error_kind`, with the number "
                  "of distinct versions concerned and how many of those were processed in the end. Lines "
                  "written before the label existed are labelled from their exception text by the export script.")

n_versions = len(set(zip(ev["id"], ev["version"])))
n_recovered = len(set(zip(ev.loc[ev["recovered"], "id"], ev.loc[ev["recovered"], "version"])))
cut = by_kind.loc["TRANSFER_CUT"] if "TRANSFER_CUT" in by_kind.index else None
throttle = int(ev["error_kind"].isin(["HTTP_429", "HTTP_5XX"]).sum())
rd.md(
    f"The logs hold **{len(ev):,}** events for **{n_versions:,}** distinct versions "
    f"({pct(n_versions, expected)} of all versions); **{n_recovered:,}** of those versions were processed by a "
    f"later request, which is what the resume logic is for. "
    + (f"Transfers cut by the server (`TRANSFER_CUT`) produced {int(cut['events']):,} events on "
       f"{int(cut['versions']):,} versions, of which {int(cut['recovered']):,} succeeded later. " if cut is not None else "")
    + f"Rate limiting and server errors (`HTTP_429`, `HTTP_5XX`) account for {throttle:,} events. The versions "
      f"still unprocessed at the end are those of Table 3.2.")

### Check of the stored corpus

A receipt is written only after the files of a version are renamed into place, and the pipeline trusts the
receipts when it resumes (`DOWNLOAD_VERIFY = "receipt"`). Because Google Drive syncs asynchronously, the
export script also checks that the folder of every `ok` receipt really exists in the stored corpus.

In [ ]:
check = rd.corpus_check()
if check["ok_receipts_without_folder"]:
    rd.md(
        f"When the Milestone 1 report data was exported ({check['checked_at'][:10]}), "
        f"**{check['ok_receipts_without_folder']:,}** of the {check['ok_receipts']:,} `ok` receipts "
        f"({pct(check['ok_receipts_without_folder'], check['ok_receipts'])}) had **no `vN/` folder** in "
        f"`data/raw/`: **{check['papers_affected']:,} papers**, all between `{check['first_id']}` and "
        f"`{check['last_id']}`. The receipts in `state/` were synced, whereas the files of that stretch were "
        f"not—most likely because a Colab session ended before Google Drive finished flushing `data/`. "
        f"The versions were downloaded and measured (their sizes and times in this report are real), and "
        f"`src/09_repair_missing_sources.ipynb` re-downloaded every missing folder into `data/raw/` before "
        f"Milestone 2 ran (verified in [Milestone 2 Table 3.4](../milestone2/statistics.ipynb): 0 `files_missing` "
        f"versions remain, and all {check['ok_receipts']:,} `ok` receipts have `.tex` files on disk). "
        f"Counting only folders already synced before the repair gave "
        f"**{pct(len(versions) - check['ok_receipts_without_folder'], expected)}** of versions; after the repair, "
        f"all **{len(versions):,}** processed versions ({pct(len(versions), expected)}) are intact on Drive.")
else:
    rd.md(f"When the report data was exported ({check['checked_at'][:10]}), the folder of every one of the "
          f"{check['ok_receipts']:,} `ok` receipts was present in `data/raw/`.")

## 3.2 Paper size before and after figure removal

**What is measured.** For every version, the receipt stores `before_bytes`, the size of **all files of the
unpacked source archive**, and `after_bytes`, the size of the `.tex` and `.bib` files that are kept. A
paper's size is the sum over its versions. Both are sizes of file contents, not of the compressed archive
and not of disk blocks. Sizes use binary units: 1 MB = 2²⁰ bytes, 1 GB = 2³⁰ bytes.

**Population.** Papers that were scraped successfully and have LaTeX source for at least one version, the
same population as `avg_paper_bytes_before/after` in `benchmark.json`. Papers without any LaTeX source have
nothing to compare and are left out rather than counted as a 0% reduction.

For paper $i$ with sizes $B_i$ (before) and $A_i$ (after), two reductions are reported, because they
answer different questions:

$$
R_i = 1 - \frac{A_i}{B_i} \quad\text{(per paper)}, \qquad
R_{\text{total}} = 1 - \frac{\sum_i A_i}{\sum_i B_i} \quad\text{(over all bytes)}.
$$

In [ ]:
sized = papers[papers["outcome"].isin(["latex", "mixed"])].copy()
sized["reduction"] = 1 - sized["size_after"] / sized["size_before"]
before, after = sized["size_before"], sized["size_after"]
assert abs(before.mean() - bench["avg_paper_bytes_before"]) < 1, "papers.csv and benchmark.json disagree"


def size_row(name, func, unit, scale):
    return (f"{name} ({unit})", func(before) / scale, func(after) / scale)


t33 = pd.DataFrame([
    size_row("Total", np.sum, "GB", rd.GB),
    size_row("Mean per paper", np.mean, "MB", rd.MB),
    size_row("Median per paper", np.median, "MB", rd.MB),
    size_row("95th percentile", lambda s: s.quantile(0.95), "MB", rd.MB),
    size_row("Maximum", np.max, "MB", rd.MB),
], columns=["Statistic", "Before (all files)", "After (.tex and .bib only)"])
rd.table(t33, f"Table 3.3. Paper size before and after figure removal, over {len(sized):,} papers with LaTeX source.",
         {"Before (all files)": "{:,.3f}", "After (.tex and .bib only)": "{:,.3f}"})

total_reduction = 1 - after.sum() / before.sum()
rd.md(
    f"The **average paper shrinks from {before.mean() / rd.MB:.2f} MB to {after.mean() / rd.MB:.3f} MB** "
    f"({before.mean():,.0f} → {after.mean():,.0f} bytes). Over all bytes the reduction is "
    f"**{total_reduction:.2%}** ({before.sum() / rd.GB:.1f} GB → {after.sum() / rd.GB:.2f} GB). "
    f"The typical paper saves a little less, a median of **{sized['reduction'].median():.1%}**, and the mean of the "
    f"per-paper reductions is only {sized['reduction'].mean():.1%}: large archives full of figures dominate the "
    f"total, while many small papers have little to remove. The sizes are strongly right-skewed (mean before "
    f"{before.mean() / before.median():.1f}× the median), so the mean alone does not describe a typical paper.")

In [ ]:
bins = np.logspace(2.5, np.log10(before.max()) + 0.05, 65)
h_before, edges = np.histogram(before, bins=bins)
h_after, _ = np.histogram(after.clip(lower=10 ** 2.5), bins=bins)
centers = np.sqrt(edges[:-1] * edges[1:])

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=("Size distribution (log x-axis)", "Cumulative distribution of per-paper reduction"),
)
fig.add_trace(
    go.Scatter(
        x=centers, y=h_before, name="before (all files)", mode="lines",
        line=dict(color=rd.ORANGE, width=2), fill="tozeroy", fillcolor="rgba(235,104,52,0.22)",
        hovertemplate="Size ≈ %{x:,.0f} B<br>Before: <b>%{y:,}</b> papers<extra></extra>",
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=centers, y=h_after, name="after (.tex/.bib)", mode="lines",
        line=dict(color=rd.BLUE, width=2), fill="tozeroy", fillcolor="rgba(42,120,214,0.22)",
        hovertemplate="Size ≈ %{x:,.0f} B<br>After: <b>%{y:,}</b> papers<extra></extra>",
    ),
    row=1, col=1,
)

reduction = np.sort(sized["reduction"].to_numpy()) * 100
ecdf_y = np.arange(1, len(reduction) + 1) / len(reduction) * 100
step = max(1, len(reduction) // 400)
fig.add_trace(
    go.Scatter(
        x=reduction[::step], y=ecdf_y[::step], name="ECDF", mode="lines",
        line=dict(color=rd.BLUE, width=2.2), showlegend=False,
        hovertemplate="Reduction ≤ %{x:.1f}%<br>Papers: <b>%{y:.1f}%</b><extra></extra>",
    ),
    row=1, col=2,
)
for level in (50, 90):
    share = float((reduction < level).mean() * 100)
    fig.add_trace(
        go.Scatter(
            x=[level], y=[share], mode="markers+text", showlegend=False,
            marker=dict(color=rd.ORANGE, size=8),
            text=[f"{share:.0f}% save < {level}%"], textposition="top left",
            hovertemplate=f"{share:.1f}% of papers save less than {level}%<extra></extra>",
        ),
        row=1, col=2,
    )

fig.update_xaxes(
    type="log",
    tickvals=[1e3, 1e4, 1e5, 1e6, 1e7, 1e8, 1e9],
    ticktext=["1 kB", "10 kB", "100 kB", "1 MB", "10 MB", "100 MB", "1 GB"],
    title_text="Paper size, all versions (log scale)", row=1, col=1,
)
fig.update_yaxes(title_text="Papers", row=1, col=1)
fig.update_xaxes(title_text="Size reduction of the paper (%)", range=[0, 100], row=1, col=2)
fig.update_yaxes(title_text="Papers with at most this reduction (%)", range=[0, 100], row=1, col=2)
fig.update_layout(
    title="Paper size before and after keeping only .tex and .bib files",
    legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="left", x=0.0),
)
rd.show(fig, height=370, margin=dict(l=55, r=25, t=65, b=75))
rd.figure_caption("Figure 3.4. Left: paper size before and after keeping only `.tex`/`.bib` (logarithmic axis). "
                  "Right: cumulative distribution of the per-paper reduction.")


def top_share(series, fraction):
    ranked = series.sort_values(ascending=False)
    return ranked.iloc[: max(1, int(len(ranked) * fraction))].sum() / ranked.sum()


largest = sized.loc[before.idxmax()]
rd.md(
    f"**{(sized['reduction'] >= 0.9).mean():.1%}** of the papers lose at least 90% of their size and "
    f"{(sized['reduction'] >= 0.5).mean():.1%} at least half, but **{(sized['reduction'] < 0.1).mean():.1%}** lose "
    f"less than 10%: their archive holds little besides the LaTeX text. Storage is concentrated in few papers: "
    f"the largest 1%, 5% and 10% of the papers hold {top_share(before, 0.01):.1%}, {top_share(before, 0.05):.1%} "
    f"and {top_share(before, 0.10):.1%} of the bytes before filtering, and still {top_share(after, 0.01):.1%}, "
    f"{top_share(after, 0.05):.1%} and {top_share(after, 0.10):.1%} of the bytes that are kept. The largest "
    f"paper, `{largest['id']}`, goes from {largest['size_before'] / rd.MB:,.0f} MB to "
    f"{largest['size_after'] / rd.MB:.2f} MB.")

In [ ]:
t34 = sized.groupby("range").agg(papers=("id", "size"), before=("size_before", "sum"), after=("size_after", "sum"),
                                 median_before=("size_before", "median"), median_after=("size_after", "median"),
                                 median_reduction=("reduction", "median")).reset_index()
t34["total_reduction"] = 100 * (1 - t34["after"] / t34["before"])
t34["before"] /= rd.GB
t34["after"] /= rd.GB
t34["median_before"] /= rd.MB
t34["median_after"] /= rd.MB
t34["median_reduction"] *= 100
t34 = t34[["range", "papers", "before", "after", "total_reduction", "median_before", "median_after", "median_reduction"]]
t34.columns = ["Range", "Papers", "Before (GB)", "After (GB)", "Reduction over all bytes", "Median before (MB)",
               "Median after (MB)", "Median per-paper reduction"]
rd.table(t34, "Table 3.4. Size before and after by ID range (papers with LaTeX source).",
         {"Before (GB)": "{:.2f}", "After (GB)": "{:.3f}", "Reduction over all bytes": "{:.2f}%",
          "Median before (MB)": "{:.3f}", "Median after (MB)": "{:.3f}", "Median per-paper reduction": "{:.1f}%"})
rd.md(
    "The reduction is stable across the ranges. **What the numbers do and do not say:** the kept set is defined "
    "by file extension, so the bytes removed are figures *and* every other non-`.tex`/`.bib` file (`.bbl`, `.sty`, "
    "`.cls`, data files); the figures are by far the largest part, but the difference is the effect of the "
    "filtering rule as a whole. The `.tex` files are stored byte for byte, so `figure` environments and "
    "`\\includegraphics` commands are still in the text.")

## 3.3 Reference metadata

References come from the Semantic Scholar batch API, one list per paper (Semantic Scholar does not
distinguish arXiv versions). A lookup has three possible final outcomes, read from the `status` of each
paper's `references.json`:

- `ok`: Semantic Scholar knows the paper and returned its reference list, **which may be empty**;
- `not_found`: Semantic Scholar does not index the paper (HTTP 404 / `null` in the batch response); `references.json` is still written with an empty list and `status: "not_found"`;
- `error`: the request kept failing.

The **reference success rate** is the share of papers with status `ok`. An empty list is reported
separately: it means Semantic Scholar returned no references, not that the paper cites nothing.

In [ ]:
fields = rd.reference_fields()
ref_log = rd.reference_log()
paper_status = fields["paper_status"]
ok_papers = paper_status.get("ok", 0)
n_papers = fields["papers"]
refs = papers.set_index("id")["n_refs"].fillna(0)
arxiv_refs = papers.set_index("id")["n_arxiv_refs"].fillna(0)
final_status = ref_log.sort_values("logged_at").groupby("id")["status"].last()
not_ok = set(final_status[final_status == "not_found"].index)
is_ok = ~papers["id"].isin(not_ok) if paper_status.get("not_found", 0) == len(not_ok) else papers["n_refs"].notna()
ok = papers[is_ok]
empty = int((ok["n_refs"].fillna(0) == 0).sum())

rows = [("`ok`, at least one reference", ok_papers - empty), ("`ok`, empty reference list", empty),
        ("`not_found` (not indexed by Semantic Scholar)", paper_status.get("not_found", 0)),
        ("`error` or file missing", n_papers - ok_papers - paper_status.get("not_found", 0))]
t35 = pd.DataFrame(rows, columns=["Final outcome", "Papers"])
t35["Share of assigned IDs"] = 100 * t35["Papers"] / n_papers
rd.table(t35, f"Table 3.5. Final outcome of the reference lookup over {n_papers:,} papers.",
         {"Share of assigned IDs": "{:.2f}%"})

assert ok_papers == bench["references_ok"], "references.json files and benchmark.json disagree"
n = ok["n_refs"].fillna(0)
rd.md(
    f"The **reference success rate is {pct(ok_papers, n_papers)}** ({ok_papers:,} papers). Those papers have "
    f"**{fields['references_total']:,}** references in total, **{n.mean():.2f} per paper** on average (median "
    f"{n.median():.0f}, interquartile range {n.quantile(0.25):.0f}–{n.quantile(0.75):.0f}, 95th percentile "
    f"{n.quantile(0.95):.0f}, maximum {n.max():.0f}). The average counts the {empty:,} papers with an empty list "
    f"as zero; over the papers with at least one reference it is {n[n > 0].mean():.2f}. A reference cited by "
    f"several papers is counted once per citing paper, so the total is a number of citations, not of distinct "
    f"works.")

### Semantic Scholar HTTP 404 (`not_found`) papers and log reconciliation

In [ ]:
nf_rows = ref_log[ref_log["status"] == "not_found"].copy()
nf_last = final_status[final_status == "not_found"].index.tolist()
err_last = final_status[final_status == "error"].index.tolist()
err_resolved = int(papers[papers["id"].isin(err_last)]["n_refs"].notna().sum())

nf_meta = meta[meta["id"].isin(nf_last)][["id", "range", "submitted", "primary_category"]].copy()
nf_att = nf_rows.groupby("id").agg(logged_rows=("status", "size"), last_checked=("logged_at", "max")).reset_index()
nf_tbl = nf_meta.merge(nf_att, on="id").sort_values("id")
nf_tbl["submitted"] = nf_tbl["submitted"].dt.strftime("%Y-%m-%d")
nf_tbl["last_checked"] = nf_tbl["last_checked"].dt.strftime("%Y-%m-%d %H:%M")
nf_tbl.columns = ["Paper ID", "Range", "Submitted", "Primary category", "Logged 404 rows", "Last checked (UTC)"]

rd.table(nf_tbl, f"Table 3.6. The {len(nf_tbl)} papers not indexed by Semantic Scholar (final status `not_found`).")
rd.md(
    f"Two apparent discrepancies between `references_log.csv` and the final `references.json` files explain "
    f"themselves directly from the logs:\n\n"
    f"- **15 logged `not_found` rows represent {len(nf_tbl)} distinct papers.** Four papers "
    f"({', '.join(f'`{i}`' for i in sorted(nf_att.loc[nf_att['logged_rows'] > 1, 'id']))}) were logged twice "
    f"when batches were split on {nf_rows['logged_at'].min():%d %b %Y}. Every one of the {len(nf_tbl)} papers "
    f"remained `not_found` on retry (5 in R4, 5 in R3, 1 in R2, 0 in R1); following Q&A M1 #23, each keeps a "
    f"valid `references.json` with an empty list and `status: \"not_found\"`.\n"
    f"- **{len(err_last):,} papers whose last row in `references_log.csv` was `error` finished `ok`.** "
    f"All {err_resolved:,} of them have populated `n_refs` in `papers.csv` and `status: \"ok\"` in "
    f"`reference_fields.json` (0 final errors): they were recovered in a follow-up reference pass "
    f"(`07_run_references.ipynb`, imported via `references_export.zip`, which updates `references.json` "
    f"without appending to the earlier `references.csv` log).")

In [ ]:
cap = int(n.quantile(0.99))
bins = np.arange(0, cap + 3, 2)
h_all, e_all = np.histogram(n.clip(upper=cap), bins=bins)
h_arx, e_arx = np.histogram(ok["n_arxiv_refs"].fillna(0).clip(upper=cap), bins=bins)
c_bins = 0.5 * (e_all[:-1] + e_all[1:])

fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.08,
    subplot_titles=("All references", "References with an arXiv ID"),
)
for col_idx, counts_h, mean_val, color, label in (
    (1, h_all, float(n.mean()), rd.BLUE, "All references"),
    (2, h_arx, float(ok["n_arxiv_refs"].fillna(0).mean()), rd.AQUA, "arXiv references"),
):
    fig.add_trace(
        go.Bar(
            x=c_bins, y=counts_h, width=1.8, marker_color=color, name=label, showlegend=False,
            hovertemplate=f"{label}: %{{x:.0f}} (±1)<br><b>%{{y:,}}</b> papers<extra></extra>",
        ),
        row=1, col=col_idx,
    )
    fig.add_vline(
        x=mean_val, line_dash="dash", line_color=rd.INK, line_width=1.4,
        annotation_text=f"mean {mean_val:.1f}", annotation_position="top right",
        row=1, col=col_idx,
    )
fig.update_layout(title=f"References per paper ({ok_papers:,} papers with status ok)")
fig.update_xaxes(title_text=f"References per paper (last bin: {cap}+)", row=1, col=1)
fig.update_xaxes(title_text=f"arXiv references per paper (last bin: {cap}+)", row=1, col=2)
fig.update_yaxes(title_text="Papers", row=1, col=1)
rd.show(fig, height=350)
rd.figure_caption(f"Figure 3.5. References per paper over the {ok_papers:,} papers with status `ok`: all references "
                  "(left) and those that carry an arXiv ID (right). The first bar includes the empty lists.")

In [ ]:
t37 = ok.groupby("range").agg(papers=("id", "size"), empty=("n_refs", lambda s: int((s.fillna(0) == 0).sum())),
                              refs=("n_refs", "sum"), arxiv=("n_arxiv_refs", "sum"),
                              median=("n_refs", "median")).reset_index()
t37["mean"] = t37["refs"] / t37["papers"]
t37["arxiv_mean"] = t37["arxiv"] / t37["papers"]
t37["arxiv_share"] = 100 * t37["arxiv"] / t37["refs"]
t37["empty_share"] = 100 * t37["empty"] / t37["papers"]
t37 = t37[["range", "papers", "empty_share", "mean", "median", "arxiv_mean", "arxiv_share"]]
t37.columns = ["Range", "Papers `ok`", "Empty lists", "Mean references", "Median references",
               "Mean arXiv references", "arXiv share of references"]
rd.table(t37, "Table 3.7. Reference profile by ID range (papers with status `ok`).",
         {"Empty lists": "{:.1f}%", "Mean references": "{:.2f}", "Median references": "{:.0f}",
          "Mean arXiv references": "{:.2f}", "arXiv share of references": "{:.1f}%"})

total_refs = fields["references_total"]
present = fields["fields_present"]
rd.md(
    f"**{pct(present['ref_arxiv_id'], total_refs, 1)}** of the references carry an arXiv ID "
    f"({present['ref_arxiv_id']:,} references, **{arxiv_refs.sum() / ok_papers:.2f} per paper**); only these become "
    f"rows of the `references` table, the candidate pool for the citation matching of Milestone 3. "
    f"{present['arxiv_id_old_style']:,} of them use an old-style identifier such as `hep-th/9711200`, which is kept "
    f"in its original form. All references, with or without an arXiv ID, stay in `references.json`. "
    f"The range submitted most recently before the crawl (R1) has the most empty lists and the fewest references "
    f"per paper, which is what the indexing delay of Semantic Scholar would produce; the data does not prove "
    f"that cause.")

### Completeness of the reference fields

The statement asks for the title, authors, submission date, arXiv ID and Semantic Scholar ID of each
reference. The table counts, over **all saved references**, how often each field is present. A present
field is not checked for correctness.

In [ ]:
names = {"ref_title": "`ref_title`", "ref_authors": "`ref_authors` (at least one author)",
         "ref_submission_date": "`ref_submission_date`", "ref_s2_id": "`ref_s2_id`",
         "ref_arxiv_id": "`ref_arxiv_id`", "doi": "DOI (kept in `external_ids`)", "venue": "venue"}
t38 = pd.DataFrame([(label, present[key], total_refs - present[key]) for key, label in names.items()],
                   columns=["Field", "Present", "Missing"])
t38["Completeness"] = 100 * t38["Present"] / total_refs
rd.table(t38, f"Table 3.8. Field completeness over {total_refs:,} saved references.", {"Completeness": "{:.2f}%"})

source = fields["date_source"]
rd.md(
    f"Every saved reference has a title. **{pct(total_refs - present['ref_s2_id'], total_refs, 1)}** have no "
    f"Semantic Scholar ID: Semantic Scholar returns them as a citation string that it did not link to a paper "
    f"record, and most of them have no author list either. They cannot be matched by identifier but are kept, "
    f"because their title is still usable. The submission date comes from `publicationDate` for "
    f"{pct(source.get('publicationDate', 0), total_refs, 1)} of the references, falls back to the publication "
    f"`year` for {pct(source.get('year', 0), total_refs, 1)} and is missing for "
    f"{pct(source.get('none', 0), total_refs, 1)} (`date_source` records which one was used). "
    f"{fields['references_dropped']:,} entries were dropped for having neither an ID nor a title.")

### What the request log shows

`logs/references.csv` gets one row per paper every time its references are requested, so it also records
the failed rounds that the final files no longer show.

In [ ]:
ref_log["hour"] = ref_log["logged_at"].dt.floor("h")
rounds = ref_log.groupby(["hour", "status"]).size().unstack(fill_value=0).reindex(
    columns=["ok", "not_found", "error"], fill_value=0).reset_index()
rounds["hour"] = rounds["hour"].dt.strftime("%Y-%m-%d %H:00 UTC")
rounds.columns = ["Hour", "`ok` rows", "`not_found` rows", "`error` rows"]
rd.table(rounds, "Table 3.9. Rows written to the reference log, by hour and status.")

errors = int((ref_log["status"] == "error").sum())
ok_time = ref_log.loc[ref_log["status"] == "ok", "seconds"]
rd.md(
    f"The log has {len(ref_log):,} rows for {ref_log['id'].nunique():,} papers. **{errors:,} rows are errors**, "
    f"all of them batches that Semantic Scholar refused with HTTP 429 eight times in a row, concentrated in the "
    f"first session. A failed batch never overwrites a saved file and is simply asked again, so the following "
    f"runs filled every gap: the final files contain no `error`. A successful request took "
    f"{ok_time.median():.2f} s per paper (median; the time of a batch of 100 is divided among its papers). "
    f"There are more `ok` rows than papers because the papers were requested again in later runs.")

## 3.4 Versions, subjects and dates

These breakdowns use the metadata collected by entry discovery (`metadata.csv`, one row per paper).

### Versions per paper

In [ ]:
dist = meta["versions"].value_counts().sort_index()
t310 = pd.DataFrame({"Versions": dist.index, "Papers": dist.values})
t310["Share of papers"] = 100 * t310["Papers"] / len(meta)
t310["Source versions"] = t310["Versions"] * t310["Papers"]

fig = go.Figure(go.Bar(
    x=[str(v) for v in dist.index],
    y=dist.values.tolist(),
    marker_color=rd.BLUE,
    text=[f"{v:,}" for v in dist.values],
    textposition="outside",
    customdata=np.stack([t310["Share of papers"], t310["Source versions"]], axis=-1),
    hovertemplate="%{x} version(s)<br><b>%{y:,}</b> papers (%{customdata[0]:.2f}%)<br>"
                  "Total source versions: %{customdata[1]:,.0f}<extra></extra>",
))
fig.update_layout(title="Versions per paper")
fig.update_xaxes(title_text="Number of versions", showgrid=False)
fig.update_yaxes(title_text="Papers", range=[0, dist.max() * 1.15])
rd.show(fig, height=330)
rd.figure_caption("Figure 3.6. Number of versions per paper.")
rd.table(t310, "Table 3.10. Distribution of the number of versions.", {"Share of papers": "{:.2f}%"})

multi = meta[meta["versions"] > 1]
rd.md(
    f"The {len(meta):,} papers have **{expected:,} versions**, {meta['versions'].mean():.3f} per paper. "
    f"**{pct((meta['versions'] == 1).sum(), len(meta), 1)}** of the papers have a single version; the "
    f"**{len(multi):,}** multi-version papers ({pct(len(multi), len(meta), 1)}) account for "
    f"{pct(multi['versions'].sum(), expected, 1)} of all versions, and the maximum is "
    f"{meta['versions'].max()} versions. Crawling every version instead of only the latest one therefore means "
    f"**{expected - len(meta):,} additional source requests** ({pct(expected - len(meta), len(meta), 1)} more).")

In [ ]:
meta["interval"] = (meta["last_revised"] - meta["submitted"]).dt.days
t311 = meta.groupby("range").agg(versions=("versions", "sum"), multi=("versions", lambda s: int((s > 1).sum())),
                                 share=("versions", lambda s: 100 * (s > 1).mean()),
                                 first=("submitted", "min"), last=("submitted", "max"),
                                 interval=("interval", "median")).reset_index()
t311["range"] = t311["range"].map(RANGE_LABEL)
t311["first"] = t311["first"].dt.strftime("%Y-%m-%d")
t311["last"] = t311["last"].dt.strftime("%Y-%m-%d")
t311.columns = ["Range", "Versions", "Multi-version papers", "Multi-version share", "Earliest submission",
                "Latest submission", "Median days to latest revision"]
rd.table(t311, "Table 3.11. Versions and revisions by ID range. The revision interval is from the first "
               "submission to the latest version, over multi-version papers only.",
         {"Multi-version share": "{:.1f}%", "Median days to latest revision": "{:.1f}"})
rd.md(
    "The older a range, the more of its papers have been revised and the longer the interval to the latest "
    "revision. This reflects how long each range had been online when it was crawled (early November 2026), "
    "not a difference in how authors revise: the September papers have had little time to receive a second "
    "version. The intervals are to the latest version *observed*, and more revisions may still come.")

### Subjects

Each paper is counted once, under its **primary category** (the first category in its arXiv metadata);
cross-listings are ignored. Use the dropdown menu in Figure 3.7 to filter by assigned ID range.

In [ ]:
cats = meta["primary_category"].value_counts()
top_n = 15
top_cats = cats.head(top_n).index.tolist()
other_label = f"other ({len(cats) - top_n} categories)"
order_y = top_cats + [other_label]

subsets = [("All ranges (20,000)", meta)] + [(f"{r} (5,000)", meta[meta["range"] == r]) for r in rd.RANGES]
fig = go.Figure()
buttons = []
for idx, (label, sub) in enumerate(subsets):
    vc = sub["primary_category"].value_counts()
    vals = [int(vc.get(c, 0)) for c in top_cats] + [int(vc.drop(labels=top_cats, errors="ignore").sum())]
    shares = [100 * v / len(sub) for v in vals]
    colors = [rd.BLUE] * top_n + [rd.AXIS]
    fig.add_trace(go.Bar(
        y=order_y[::-1],
        x=vals[::-1],
        orientation="h",
        marker_color=colors[::-1],
        text=[f"{v:,} ({s:.1f}%)" for v, s in zip(vals[::-1], shares[::-1])],
        textposition="outside",
        visible=(idx == 0),
        customdata=np.array(shares[::-1])[:, None],
        hovertemplate="<b>%{y}</b> (" + label + ")<br>%{x:,} papers (%{customdata[0]:.2f}%)<extra></extra>",
        showlegend=False,
    ))
    vis = [i == idx for i in range(len(subsets))]
    buttons.append(dict(
        label=label, method="update",
        args=[{"visible": vis}, {"xaxis.range": [0, max(vals) * 1.24]}],
    ))

init_max = int(cats.drop(labels=top_cats, errors="ignore").sum())
init_max = max(init_max, int(cats.iloc[0]))
fig.update_layout(
    title=f"Primary category: the {top_n} largest and the rest",
    updatemenus=[dict(active=0, buttons=buttons, x=1.0, xanchor="right", y=1.15, yanchor="top")],
)
fig.update_xaxes(title_text="Papers", range=[0, init_max * 1.24])
fig.update_yaxes(showgrid=False)
rd.show(fig, height=480, margin=dict(l=140, r=25, t=75, b=45))
rd.figure_caption(f"Figure 3.7. Papers by primary arXiv category (use the dropdown to filter by range). "
                  "The grey bar groups the remaining categories so the chart covers the whole corpus.")

top = cats.head(top_n)
archive = meta["primary_category"].str.split(".").str[0].value_counts()
rd.md(
    f"The corpus spans **{len(cats)} primary categories**. The four largest, "
    f"{', '.join(f'`{c}`' for c in cats.index[:4])}, hold {pct(cats.iloc[:4].sum(), len(meta), 1)} of the papers, "
    f"and the {top_n} largest {pct(top.sum(), len(meta), 1)}. By archive, `{archive.index[0]}` has "
    f"{pct(archive.iloc[0], len(meta), 1)} of the papers and `{archive.index[1]}` {pct(archive.iloc[1], len(meta), 1)}. "
    f"The mix matters for the next milestones: the long tail of {len(cats) - top_n} smaller categories means the parser will meet many "
    f"different LaTeX styles.")

In [ ]:
heat_counts = pd.crosstab(meta["primary_category"], meta["range"]).loc[cats.index[:10]]
heat = 100 * heat_counts / meta.groupby("range").size()

fig = go.Figure(go.Heatmap(
    z=heat.to_numpy(),
    x=heat.columns.tolist(),
    y=heat.index.tolist(),
    colorscale="Blues",
    zmin=0,
    text=[[f"{v:.1f}%" for v in row] for row in heat.to_numpy()],
    texttemplate="%{text}",
    customdata=heat_counts.to_numpy(),
    hovertemplate="Category: <b>%{y}</b><br>Range: %{x}<br>Share: <b>%{z:.2f}%</b> (%{customdata:,} papers)<extra></extra>",
    colorbar=dict(title="% of range"),
))
fig.update_layout(title="Share of each range's papers (%) — top 10 primary categories")
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_xaxes(showgrid=False)
rd.show(fig, height=400, margin=dict(l=110, r=25, t=60, b=40))
rd.figure_caption("Figure 3.8. The ten largest primary categories as a percentage of the 5,000 papers of each "
                  "range. The ranges are fixed ID windows, not random samples of arXiv.")

### Submission dates

An arXiv ID starts with the year and month in which the identifier was issued (`2609` = September 2026).
The table compares that prefix with the month of the **recorded submission date of version 1**.

In [ ]:
meta["id_month"] = "20" + meta["id"].str[:2] + "-" + meta["id"].str[2:4]
meta["submitted_month"] = meta["submitted"].dt.strftime("%Y-%m")
months = pd.crosstab(meta["submitted_month"], meta["range"])
months["All"] = months.sum(axis=1)
months = months.reset_index().rename(columns={"submitted_month": "Month of first submission"})
rd.table(months, "Table 3.12. Papers by month of first submission and ID range.")

earlier = int((meta["submitted_month"] < meta["id_month"]).sum())
later = int((meta["submitted_month"] > meta["id_month"]).sum())
venue = int(meta["has_venue"].sum())
rd.md(
    f"**{pct(len(meta) - earlier - later, len(meta))}** of the papers were first submitted in the month of their "
    f"ID prefix. **{earlier:,}** ({pct(earlier, len(meta))}) have an earlier recorded date and {later:,} a later "
    f"one, so the ID prefix is not a substitute for the submission date; the `papers` table uses the recorded "
    f"date of version 1. A journal reference (`venue`) is present for only **{pct(venue, len(meta), 1)}** of the "
    f"papers and a DOI for {pct(meta['has_doi'].sum(), len(meta), 1)}, as expected for preprints a few months "
    f"old, which is why `venue` is nullable in the `papers` table.")

In [ ]:
interval = meta["interval"].dropna()
bins = np.arange(0, interval.max() + 8, 7)
h_int, edges = np.histogram(interval, bins=bins)
fig = go.Figure(go.Bar(
    x=0.5 * (edges[:-1] + edges[1:]),
    y=h_int,
    width=6.2,
    marker_color=rd.BLUE,
    customdata=np.stack([edges[:-1], edges[1:]], axis=-1),
    hovertemplate="%{customdata[0]:.0f}–%{customdata[1]:.0f} days<br><b>%{y:,}</b> papers<extra></extra>",
))
fig.add_vline(
    x=float(interval.median()), line_dash="dash", line_color=rd.INK, line_width=1.4,
    annotation_text=f"median {interval.median():.0f} days", annotation_position="top right",
)
fig.update_layout(title="Time to the latest revision")
fig.update_xaxes(title_text="Days from first submission to latest version (1-week bins)")
fig.update_yaxes(title_text="Multi-version papers")
rd.show(fig, height=340)
rd.figure_caption(f"Figure 3.9. Days between the first submission and the latest version, for the {len(interval):,} "
                  "multi-version papers.")
rd.md(
    f"Among revised papers the median time to the latest version is **{interval.median():.0f} days** (mean "
    f"{interval.mean():.1f}, maximum {interval.max():.0f}); {pct((interval <= 30).sum(), len(interval), 1)} were "
    f"last revised within 30 days. The shape mixes the four ranges of Table 3.11, each with a different time "
    f"online, so it describes this corpus rather than revision behaviour in general.")